# AfriSenti — téléchargement, nettoyage et audit

Ce notebook prépare le corpus marocain AfriSenti pour LahjaLens. Il conserve les splits officiels, vérifie les labels et produit les fichiers propres utilisés par Spark ML.

In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import re
import unicodedata
import urllib.request
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RAW_DIR = ROOT / 'data' / 'raw' / 'afrisenti' / 'v1'
PROCESSED_DIR = ROOT / 'data' / 'processed' / 'afrisenti' / 'v1'
RAW_BASE = 'https://raw.githubusercontent.com/afrisenti-semeval/afrisent-semeval-2023/main/data/ary'
SPLITS = ('train', 'dev', 'test')
LABELS = {'negative', 'neutral', 'positive'}
OUTPUT_FIELDS = ['record_id', 'comment_text', 'sentiment', 'label', 'source_dataset', 'source_split', 'language', 'cleaning_version']

def utc_now():
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')

def sha256(payload):
    return hashlib.sha256(payload).hexdigest()

def normalize_text(value):
    value = unicodedata.normalize('NFKC', value)
    value = ''.join(char for char in value if unicodedata.category(char) != 'Cf' and (unicodedata.category(char) != 'Cc' or char in '\t\n\r'))
    return re.sub(r'\s+', ' ', value).strip()

## 1. Télécharger et vérifier les fichiers bruts

Les fichiers originaux sont conservés sans modification. Le manifeste garde les URL et les empreintes SHA-256.

In [ ]:
RAW_DIR.mkdir(parents=True, exist_ok=True)
manifest_files = {}
for split in SPLITS:
    name = f'{split}.tsv'
    url = f'{RAW_BASE}/{name}'
    request = urllib.request.Request(url, headers={'User-Agent': 'LahjaLens-afrisenti-notebook/0.1'})
    with urllib.request.urlopen(request, timeout=60) as response:
        payload = response.read()
    if not payload or payload.lstrip().startswith((b'<html', b'<!DOCTYPE')):
        raise ValueError(f'Reponse inattendue: {url}')
    target = RAW_DIR / name
    if target.exists() and target.read_bytes() != payload:
        raise ValueError(f'Le fichier local differe de la source: {target}')
    target.write_bytes(payload)
    manifest_files[name] = {'url': url, 'bytes': len(payload), 'sha256': sha256(payload)}
    print(name, len(payload), manifest_files[name]['sha256'][:12])

manifest = {'dataset': 'AfriSenti', 'version': 'v1', 'language': 'Moroccan Arabic / Darija', 'license': 'CC BY 4.0', 'source_repository': 'https://github.com/afrisenti-semeval/afrisent-semeval-2023', 'source_ref': 'main/data/ary', 'downloaded_at': utc_now(), 'files': manifest_files}
(RAW_DIR / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

## 2. Extraire le TSV et contrôler le contrat

Le schéma attendu est `tweet`, `label`. Tout label hors des trois classes est refusé.

In [ ]:
def parse_tsv(path, split):
    with path.open(encoding='utf-8-sig', newline='') as handle:
        reader = csv.DictReader(handle, delimiter='\t')
        if reader.fieldnames != ['tweet', 'label']:
            raise ValueError(f'Schema inattendu pour {split}: {reader.fieldnames}')
        rows = []
        for number, row in enumerate(reader, 2):
            label = (row.get('label') or '').strip().lower()
            if row.get('tweet') is None or label not in LABELS:
                raise ValueError(f'Ligne invalide {split}:{number}')
            rows.append({'tweet': row['tweet'], 'label': label})
    return rows

raw_rows = {split: parse_tsv(RAW_DIR / f'{split}.tsv', split) for split in SPLITS}
{split: len(rows) for split, rows in raw_rows.items()}

## 3. Nettoyer sans détruire l'information

Nous appliquons uniquement NFKC, la suppression de caractères de contrôle/format et la normalisation des espaces. Nous ne supprimons pas la négation, les emojis ou l'Arabizi.

In [ ]:
normalized = {split: [{'text': normalize_text(row['tweet']), 'label': row['label']} for row in rows if normalize_text(row['tweet'])] for split, rows in raw_rows.items()}
test_keys = {row['text'] for row in normalized['test']}
seen_previous = set()
cleaned = {split: [] for split in SPLITS}
removed = Counter()
for split in SPLITS:
    seen_current = set()
    for row in normalized[split]:
        key = row['text']
        if split != 'test' and key in test_keys:
            removed[f'{split}_overlaps_test'] += 1
            continue
        if split != 'test' and key in seen_previous:
            removed[f'{split}_overlaps_previous_split'] += 1
            continue
        if key in seen_current:
            removed[f'{split}_duplicate'] += 1
            continue
        seen_current.add(key)
        cleaned[split].append(row)
    seen_previous.update(seen_current)
removed

## 4. Écrire les données prêtes pour Spark ML

Le test officiel est gardé séparément. Le champ `label` utilise la convention `negative=0`, `neutral=1`, `positive=2`.

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
label_to_id = {'negative': 0, 'neutral': 1, 'positive': 2}
split_stats = {}
for split in SPLITS:
    output_rows = []
    for row in cleaned[split]:
        digest = hashlib.sha1(row['text'].encode('utf-8')).hexdigest()[:16]
        output_rows.append({'record_id': f'afrisenti:ary:{split}:{digest}', 'comment_text': row['text'], 'sentiment': row['label'], 'label': label_to_id[row['label']], 'source_dataset': 'afrisenti', 'source_split': split, 'language': 'darija', 'cleaning_version': 'nfkc_whitespace_v1'})
    target = PROCESSED_DIR / f'{split}.csv'
    with target.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=OUTPUT_FIELDS)
        writer.writeheader()
        writer.writerows(output_rows)
    split_stats[split] = {'rows': len(output_rows), 'labels': dict(sorted(Counter(row['sentiment'] for row in output_rows).items())), 'sha256': sha256(target.read_bytes())}

audit = {'created_at': utc_now(), 'dataset': 'AfriSenti', 'version': 'v1', 'license': 'CC BY 4.0', 'cleaning': ['UTF-8 BOM handling', 'Unicode NFKC', 'control/format character removal', 'whitespace normalization'], 'label_mapping': label_to_id, 'official_splits_preserved': True, 'removed_rows': dict(sorted(removed.items())), 'splits': split_stats, 'warning': 'AfriSenti fournit le sentiment, pas les thèmes Hespress.'}
(PROCESSED_DIR / 'audit.json').write_text(json.dumps(audit, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
audit